In [1]:
import os
import torch
import pandas as pd

from tqdm import tqdm

from stock_bpt import StockBPT, LinearModel, NaiveModel
from dataloader_builder import build_dataloaders
from setup import StockBPT_cfg, LinearModel_cfg, NaiveModel_cfg
from setup import path_data_preprocessor
from model_training import model_setup, train_model_cuda

from model_training import train_model_cuda, evaluate_model, evaluate_best_model
from model_analysis import test_model, print_loss_analysis, process_losses, format_num, process_result, store_result

In [2]:
cuda = True if torch.cuda.is_available() else False

print("PyTorch:", torch.__version__)
print("CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

PyTorch: 2.13.0+cu132
CUDA build: 13.2
CUDA available: True
GPU: NVIDIA GeForce RTX 4070 Laptop GPU


## MODEL TRAINING ---------------------------

In [3]:
torch.manual_seed(1234)
print(path_data_preprocessor)
dls, train_norms = build_dataloaders(path_data_preprocessor)

preprocessed_data/data_1min_2021_2026
Building DataLoaders...
Train dataset samples: 89,374
Train loader batches:  349
Batch size:            256


In [4]:
optimizer_data = [torch.optim.AdamW, 0.0004, 0.1]
scaler_data = [torch.amp.GradScaler, "cuda"]

max_epochs = 10

eval_bs = 1000

stockBPT, stockBPT_params, opt1, sca1, sch1 = model_setup(StockBPT, StockBPT_cfg, train_norms, device,
                                                *optimizer_data, *scaler_data)
linearModel, linearModel_params, opt2, sca2, sch2 = model_setup(LinearModel, LinearModel_cfg, train_norms, device, 
                                                      *optimizer_data, *scaler_data)
naiveModel = NaiveModel(NaiveModel_cfg, train_norms)
naiveModel.to(device)

Input Norm: torch.Size([12])|torch.Size([12])
Target Norm: torch.Size([4])|torch.Size([4])
3260416
5376


NaiveModel()

In [5]:
model_train_losses, model_val_losses = train_model_cuda(stockBPT, device, opt1, sca1, sch1, max_epochs, 
                                                        dls["train"], dls["val"], eval_bs)
linear_train_losses, linear_val_losses = train_model_cuda(linearModel, device, opt2, sca2, sch2, max_epochs,
                                                        dls["train"], dls["val"], eval_bs)


Continuing from previous checkpoint...
[] []


|          | 0.0% (00:00) Setting up...                                                                   

Epoch 11:

Finished
Continuing from previous checkpoint...
[] []


|          | 0.0% (00:00) Setting up...                                                                   

Epoch 11:

Finished


## Model Analysis -------------------------

In [6]:
#* REUSES OBJETCS FROM TRAINING
analysis_steps = min(eval_bs, len(dls["train"])) + min(eval_bs, len(dls["val"])) + min(eval_bs, len(dls["test"]))
analysis_pbar = tqdm(total=3*analysis_steps, desc=f"Evaluating the best model parameters...".ljust(80),
                bar_format="|{bar}| {percentage:3.1f}% ({elapsed}) {desc}", position=0, leave=False)

#* Reevaluates models by their best parameters on train and val dataloaders
naive_losses = evaluate_model(dls["train"], dls["val"], naiveModel, device, eval_bs, analysis_pbar)
linear_losses = evaluate_best_model(linearModel, device, opt2, sca2, sch2, dls["train"], dls["val"], eval_bs, analysis_pbar, True)
bpt_losses = evaluate_best_model(stockBPT, device, opt1, sca1, sch1, dls["train"], dls["val"], eval_bs, analysis_pbar, True) 

#* Final evaluation on unseen test dataloader
naive_test_losses = test_model(dls["test"], naiveModel, device, eval_bs, analysis_pbar)
linear_test_losses = test_model(dls["test"], linearModel, device, eval_bs, analysis_pbar)
bpt_test_losses = test_model(dls["test"], stockBPT, device, eval_bs, analysis_pbar)


|█▎        | 12.7% (00:06) Evaluating model on training data... (168/349) [169/1329]:                     

KeyboardInterrupt: 

In [ ]:
for key, features in [("NLL", StockBPT_cfg["target_features"]),
                      ("MAE", StockBPT_cfg["target_features"]),
                      ("RMAE", StockBPT_cfg["target_features"]),
                      ("STD", [f"{feature}_std" for feature in StockBPT_cfg["target_features"]]),
                      ("RSTD", [f"{feature}_std" for feature in StockBPT_cfg["target_features"]]),
                      ("Z^2", StockBPT_cfg["target_features"])]:
    print_loss_analysis(process_losses(bpt_losses + bpt_test_losses +
                                       linear_losses + linear_test_losses +
                                       naive_losses + naive_test_losses, key), 
                                       [stockBPT.cfg["name"], linearModel.cfg["name"], naiveModel.cfg["name"]],
                                       [format_num(stockBPT_params), format_num(linearModel_params), "0"], 
                                       features, key)


--------------------------------------------------------------------------------------------------------------

NLL

--------------------------------------------------------------------------------------------------------------

                    o        h        l        c        
StockGPT-v15-2.25-0.75_50: 3.3M
    Training:       0.7818   0.7627   0.7762   0.7783     >  0.7748
    Validation:     0.7678   0.7521   0.7660   0.7678     >  0.7634
    Testing:        0.7104   0.6945   0.7068   0.7080     >  0.7049
    
LinearModel-v15-2.25-0.75_50: 5.4K
    Training:       0.9205   0.9083   0.9248   0.9253     >  0.9197
    Validation:     0.8778   0.8675   0.8832   0.8830     >  0.8779
    Testing:        0.8401   0.8317   0.8462   0.8453     >  0.8408
    
NaiveModel-B1_50: 0
    Training:       1.7124   1.7086   1.7144   1.7118     >  1.7118
    Validation:     1.6892   1.6858   1.6918   1.6884     >  1.6888
    Testing:        1.7068   1.7035   1.7093   1.7053     >  1.7062
    

In [10]:
dls, train_norms = build_dataloaders("handpicked_data", False, drop_last = False)

Building DataLoaders...


In [11]:
test_losses = test_model(dls["test"], stockBPT, device, eval_bs)
print(test_losses)

({'NLL': tensor([1.6715, 1.6525, 1.6404, 1.6757], device='cuda:0'), 'MAE': tensor([1.1373, 1.1040, 1.0898, 1.1246], device='cuda:0'), 'RMAE': tensor([0.0162, 0.0160, 0.0155, 0.0162], device='cuda:0'), 'STD': tensor([2.9651, 2.9094, 2.9305, 2.9481], device='cuda:0'), 'RSTD': tensor([0.0422, 0.0422, 0.0417, 0.0424], device='cuda:0'), 'Z^2': tensor([0.2765, 0.3003, 0.2697, 0.2868], device='cuda:0')},)
